In [7]:
import sys
sys.path.append('..')
%load_ext autoreload
%autoreload 2
import pandas as pd

trip = 'PVS1'
frames = []
for side in ['left', 'right']:
    w = pd.read_csv(f'../data/processed/windows_metrics_{trip}_{side}.csv')
    w['trip_id'] = trip
    w['side'] = side
    frames.append(w)

all_windows = pd.concat(frames, ignore_index = True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
vehicle = 'Volkswagen Saveiro'
driver = 'Driver 1'
scenario = 'Scenario 1'

trips = pd.DataFrame([{'trip_id': trip, 'vehicle': vehicle, 'driver': driver, 'scenario': scenario}])
windows_tbl = all_windows[['trip_id', 'side', 'window', 'surface', 't_start', 'n_rows', 'speed_mean', 'speed_std', 'valid']]
positions = ['dashboard', 'above_suspension']
metric_names = {'aw': 'aw', 'rms': 'rms', 'cf': 'crest_factor', 'vdv': 'vdv'}

frames = []
for pos in positions:
    cols = {f'{m}_{pos}': name for m, name in metric_names.items() if f'{m}_{pos}' in all_windows.columns}
    m = all_windows[['trip_id', 'side', 'window'] + list(cols)].rename(columns = cols)
    m['position'] = pos
    frames.append(m)

metrics = pd.concat(frames, ignore_index = True).dropna(subset = ['aw'])

In [9]:
import sqlite3
from src.db import build_database

db = '../data/processed/ride.db'
build_database(db, {'trips': trips, 'windows': windows_tbl, 'metrics': metrics})

con = sqlite3.connect(db)
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con))
print(pd.read_sql("SELECT COUNT(*) AS n FROM metrics", con))

      name
0    trips
1  windows
2  metrics
     n
0  850


In [10]:
q1 = """
SELECT w.surface,
       m.position,
       COUNT(*)         AS n_windows,
       AVG(m.aw)        AS aw_mean,
       AVG(m.rms)       AS rms_mean
FROM metrics AS m
JOIN windows AS w
  ON  m.trip_id = w.trip_id
  AND m.side    = w.side
  AND m.window  = w.window
GROUP BY w.surface, m.position
ORDER BY m.position, aw_mean DESC;
"""
display(pd.read_sql(q1, con).round(3))

,surface,position,n_windows,aw_mean,rms_mean
0,dirt_road,above_suspension,83,1.912,2.432
1,cobblestone_road,above_suspension,192,1.854,2.502
2,asphalt_road,above_suspension,150,0.833,1.122
3,cobblestone_road,dashboard,192,1.308,1.958
4,dirt_road,dashboard,83,1.277,1.774
5,asphalt_road,dashboard,150,0.583,0.829
